In [ ]:
# import os
# import sys

# # 1. Search the entire Kaggle input directory for your specific file
# target_file = 'aies_dataloader.py'
# found_path = None

# for root, dirs, files in os.walk('/kaggle/input'):
#     if target_file in files:
#         found_path = root
#         break

# if found_path:
#     print(f"🎯 Found dataset folder at: {found_path}")
    
#     # 2. Add that exact folder to Python's path
#     if found_path not in sys.path:
#         sys.path.append(found_path)
    
#     # 3. Now import your classes
#     from aies_dataloader import CGCD_Master_Dataset, DATA_PATHS
#     print("✅ Data Engine Successfully Imported!")
# else:
#     print(f"❌ Error: Could not find {target_file}. Please check the right sidebar to ensure the dataset is attached.")

## CELL 1: Environment & Dataset Router

In [ ]:
import os
import sys
from unittest.mock import MagicMock
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# CELL 1: ENVIRONMENT & MOCK HUB (Ensuring Exact Same Data Access)
# ==============================================================================

def fake_dataset_download(handle):
    h = handle.lower()
    if "cub" in h: return "/kaggle/input/datasets/wenewone/cub2002011/CUB_200_2011"
    if "tiny" in h: return "/kaggle/input/datasets/akash2sharma/tiny-imagenet/tiny-imagenet-200/tiny-imagenet-200"
    if "imagenet100" in h: return "/kaggle/input/datasets/ambityga/imagenet100"
    return "/kaggle/working/data"

mock_hub = MagicMock()
mock_hub.dataset_download = fake_dataset_download
sys.modules['kagglehub'] = mock_hub

found_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    for ignore in ['tiny-imagenet', 'cub2002011', 'imagenet100', 'images']:
        if ignore in dirs: dirs.remove(ignore) 
    if 'aies_dataloader.py' in files:
        found_path = root
        break

if found_path:
    if found_path not in sys.path: sys.path.append(found_path)
    print(f"🎯 Found dataloader script at: {found_path}")
else:
    raise FileNotFoundError("❌ Script not found. Check the right sidebar.")

from aies_dataloader import CGCD_Master_Dataset, DATA_PATHS
print("✅ Baseline Dataloader imported successfully!")

## CELL 2: Dataloader Wrapper

In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image

# ==============================================================================
# CELL 2: SINGLE-VIEW DATASET WRAPPER
# ==============================================================================
# Your method evaluates instances geometrically, so we only need 1 standard view
# instead of the 2 heavily augmented contrastive views Happy-CGCD uses.

class GeometricDatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        real_idx = self.base_dataset.final_indices[idx]
        
        if self.base_dataset.dataset_name == "C100":
            image = self.base_dataset.image_paths_or_data[real_idx]
            image = Image.fromarray(image)
        else:
            img_path = self.base_dataset.image_paths_or_data[real_idx]
            image = Image.open(img_path).convert('RGB')
            
        label = self.base_dataset.all_targets[real_idx]
        image_t = self.transform(image)
        
        return image_t, label, real_idx

standard_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

print("✅ Geometric Dataset Wrapper Initialized!")

## Neural Architecture & Margin Losses

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# ==============================================================================
# CELL 3: DINO ARCHITECTURE & YOUR GEOMETRIC LOSSES
# ==============================================================================

class DINOHead(nn.Module):
    def __init__(self, in_dim, out_dim, use_bn=False, norm_last_layer=True, nlayers=3, hidden_dim=2048, bottleneck_dim=256):
        super().__init__()
        nlayers = max(nlayers, 1)
        if nlayers == 1:
            self.mlp = nn.Linear(in_dim, bottleneck_dim)
        else:
            layers = [nn.Linear(in_dim, hidden_dim)]
            if use_bn: layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.GELU())
            for _ in range(nlayers - 2):
                layers.append(nn.Linear(hidden_dim, hidden_dim))
                if use_bn: layers.append(nn.BatchNorm1d(hidden_dim))
                layers.append(nn.GELU())
            layers.append(nn.Linear(hidden_dim, bottleneck_dim))
            self.mlp = nn.Sequential(*layers)
        self.apply(self._init_weights)
        self.last_layer = nn.utils.weight_norm(nn.Linear(bottleneck_dim, out_dim, bias=False))
        self.last_layer.weight_g.data.fill_(1)
        if norm_last_layer:
            self.last_layer.weight_g.requires_grad = False

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            torch.nn.init.trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x_proj = self.mlp(x)
        x_norm = nn.functional.normalize(x_proj, dim=-1, p=2)
        logits = self.last_layer(x_norm)
        return x_norm, logits

# --- YOUR LOSS FUNCTION ---
def margin_contrastive_loss(z, labels, pos_margin=0.85, neg_margin=0.1):
    sim = torch.matmul(z, z.T)
    mask = torch.eye(z.shape[0], dtype=torch.bool, device=z.device)
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask
    labels_diff = ~labels_equal & ~mask
    pos_loss = F.relu(pos_margin - sim)[labels_equal].mean() if labels_equal.sum() > 0 else torch.tensor(0.0)
    neg_loss = F.relu(sim - neg_margin)[labels_diff].mean() if labels_diff.sum() > 0 else torch.tensor(0.0)
    return pos_loss + neg_loss

print("✅ Architecture & Losses Loaded!")

## Memory Buffer & OOD Engine

In [ ]:
import numpy as np
from collections import defaultdict

# ==============================================================================
# CELL 4: MEMORY HERDING & OOD DETECTION
# ==============================================================================

class MemoryBuffer:
    def __init__(self, max_per_class=400):
        self.data = defaultdict(list)
        self.max_per_class = max_per_class

    @torch.no_grad()
    def build_memory_herding(self, X_all, y_label, model, device):
        model.eval()
        logits_all, Z_all = [], []
        for i in range(0, len(X_all), 128):
            batch = X_all[i:i+128].to(device)
            z_norm, logits = model[1](model[0](batch)) # DINO -> Head
            logits_all.append(logits.cpu())
            Z_all.append(z_norm.cpu())
            
        logits_all = torch.cat(logits_all)
        Z_all = torch.cat(Z_all)
        class_mean = F.normalize(Z_all.mean(0), dim=0)
        selected_idx, features = [], Z_all.clone()

        for k in range(min(self.max_per_class, len(X_all))):
            if k > 0: S = Z_all[selected_idx].sum(0)
            else: S = torch.zeros_like(class_mean)
            target = (k + 1) * class_mean - S
            distances = torch.norm(features - target, dim=1)
            for idx in selected_idx: distances[idx] = float('inf')
            best = distances.argmin().item()
            selected_idx.append(best)

        self.data[int(y_label)] = []
        for idx in selected_idx:
            self.data[int(y_label)].append((X_all[idx].detach().cpu(), logits_all[idx].detach().cpu()))

    def sample_balanced(self, batch_size):
        classes = list(self.data.keys())
        if not classes: return None, None
        samples_per_class = max(1, batch_size // len(classes))
        X_mem, Y_mem = [], []
        for cls in classes:
            samples = self.data[cls]
            if not samples: continue
            replace = len(samples) < samples_per_class
            idx = np.random.choice(len(samples), samples_per_class, replace=replace)
            for i in idx:
                X_mem.append(samples[i][0])
                Y_mem.append(cls)
        if not X_mem: return None, None
        return torch.stack(X_mem), torch.tensor(Y_mem)

class HypersphereNovelty:
    def __init__(self, q=0.90):
        self.q = q
        self.mu, self.r = {}, {}

    def update(self, memory, model, device):
        self.mu, self.r = {}, {}
        for k, X_tuples in memory.data.items():
            if len(X_tuples) == 0: continue
            X = torch.stack([x for x, _ in X_tuples]).to(device)
            with torch.no_grad(): 
                z_norm, _ = model[1](model[0](X))
            mu = F.normalize(z_norm.mean(0), dim=0)
            d = 1 - torch.matmul(z_norm, mu)
            self.mu[k] = mu
            self.r[k] = torch.quantile(d, self.q)

    def score(self, z):
        if not self.mu: return torch.tensor(0.0)
        return min([(1 - torch.dot(z, self.mu[k].to(z.device))) - self.r[k].to(z.device) for k in self.mu])

print("✅ Memory & OOD Engines Loaded!")

## Full Evaluation Pipeline

In [ ]:
# from torch.optim import SGD
# from sklearn.cluster import KMeans
# from sklearn.neighbors import NearestNeighbors
# from scipy.optimize import linear_sum_assignment
# import scipy.spatial.distance as sp_dist
# import hdbscan
# import copy
# import numpy as np
# import torch
# import torch.nn.functional as F
# from PIL import Image
# from collections import Counter

# # ==============================================================================
# # CELL 5: THE YOUR-METHOD PIPELINE (Apples-to-Apples with Happy-CGCD)
# # ==============================================================================
# DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# # ---> 🎛️ THE MASTER EXECUTION TOGGLE 🎛️ <---
# RUN_MODE = "1.5_HOUR_TEST"  # Options: "FAST_TEST", "2_HOUR_TEST", "FULL_RUN"
# TARGET_DATASET = "C100" 

# # --- HUNGARIAN MATCHING ALGORITHM (From Baseline) ---
# def cluster_acc(y_true, y_pred, mask):
#     y_true, y_pred, mask = y_true.astype(int), y_pred.astype(int), mask.astype(bool)
#     D = max(y_pred.max(), y_true.max()) + 1
#     w = np.zeros((D, D), dtype=int)
#     for i in range(y_pred.size): w[y_pred[i], y_true[i]] += 1
#     ind = np.vstack(linear_sum_assignment(w.max() - w)).T
#     ind_map = {j: i for i, j in ind}
#     total_acc = sum([w[i, j] for i, j in ind]) * 1.0 / y_pred.size
    
#     old_classes_gt, new_classes_gt = set(y_true[mask]), set(y_true[~mask])
#     old_acc, total_old = 0, 0
#     for i in old_classes_gt:
#         old_acc += w[ind_map[i], i] if i in ind_map else 0
#         total_old += sum(w[:, i])
#     old_acc = (old_acc / total_old) * 100 if total_old > 0 else 0.0

#     new_acc, total_new = 0, 0
#     for i in new_classes_gt:
#         new_acc += w[ind_map[i], i] if i in ind_map else 0
#         total_new += sum(w[:, i])
#     new_acc = (new_acc / total_new) * 100 if total_new > 0 else 0.0
#     return total_acc * 100, old_acc, new_acc

# # DYNAMIC PARAMETERS (Adjusted for 90% Sparsity Starvation) ---
# #v1
# # ALPHA = 0.50          
# # BETA = 0.01           
# # DELTA = 5             
# # MIN_CLUSTER_SIZE = 5     
# # PROMOTION_THRESHOLD = 20 

# #v2
# # --- TWEAK THESE FOR 90% SPARSITY ---
# # ALPHA = 0.60             # (Relaxed from 0.50) Allow slightly looser novel clusters
# # BETA = 0.01              # <--- ADD THIS LINE BACK IN!
# # DELTA = 3                # (Relaxed from 5) Require less density to pass the gate
# # MIN_CLUSTER_SIZE = 3     # (Relaxed from 5) Allow tiny HDBSCAN clusters to form
# # PROMOTION_THRESHOLD = 8  # (Relaxed from 20) Promote as soon as we stash 8 safe images!

# #v3
# ALPHA = 0.55             # Tighten the cluster requirement slightly
# BETA = 0.01              # <--- ADD THIS LINE BACK IN!
# DELTA = 4                # Require slightly more density
# MIN_CLUSTER_SIZE = 4     
# PROMOTION_THRESHOLD = 12 # The sweet spot for 90% sparsity!


# MNN_K = 5                
# MERGE_THRESHOLD = 0.90   

# DATASET_CONFIGS = {"C100": {"base": 50, "chunk": 10}, "IN100": {"base": 50, "chunk": 10}, "TINY": {"base": 100, "chunk": 20}, "CUB": {"base": 100, "chunk": 20}}
# NUM_BASE_CLASSES = DATASET_CONFIGS[TARGET_DATASET]["base"]

# if RUN_MODE == "2_HOUR_TEST": 
#     EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 10, 5, 999999
# if RUN_MODE == "1.5_HOUR_TEST": 
#     EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 5, 3, 999999
# elif RUN_MODE == "FAST_TEST": 
#     EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 1, 1, 3
# else: 
#     EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 100, 30, 999999

# # Init DINO
# print("Initializing DINO ViT Backbone...")
# backbone = torch.hub.load('facebookresearch/dino:main', 'dino_vitb16', verbose=False)
# for m in backbone.parameters(): m.requires_grad = False
# for name, m in backbone.named_parameters():
#     if 'blocks.11' in name: m.requires_grad = True

# model = nn.Sequential(backbone, DINOHead(in_dim=768, out_dim=NUM_BASE_CLASSES)).to(DEVICE)
# memory = MemoryBuffer(max_per_class=100)
# detector = HypersphereNovelty()

# stage_metrics = {}
# discovery_log = [] 
# noise_log = []     
# purity_log = []
# sparsity_level_final = 0.90
# print("\n" + "="*80)
# print(f"🚀 STARTING YOUR GEOMETRIC OOD PIPELINE: {RUN_MODE}")
# print(f"⚙️ Config Locked: Offline Epochs: {EPOCHS_OFFLINE} | Online Epochs: {EPOCHS_ONLINE} | Sparsity level: {sparsity_level_final}")
# print(f"Sparsity Logic: MIN_CLUSTER={MIN_CLUSTER_SIZE}, PROMOTE_THR={PROMOTION_THRESHOLD}")
# print("="*80)

# for stage in range(6):
#     print(f"\n---> INITIALIZING STAGE {stage} <---")
#     ds = CGCD_Master_Dataset(TARGET_DATASET, DATA_PATHS[TARGET_DATASET], stage=stage, sparsity_level=0.90)
#     loader = torch.utils.data.DataLoader(GeometricDatasetWrapper(ds, standard_transform), batch_size=128, drop_last=False, shuffle=True)
    
#     if stage == 0:
#         print("🎓 [STAGE 0] Training Base Classes...")
#         model.train()
#         opt = SGD([{'params': [p for p in model.parameters() if p.requires_grad]}], lr=0.1, momentum=0.9, weight_decay=5e-5)
#         sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_OFFLINE)
        
#         for epoch in range(EPOCHS_OFFLINE):
#             for b_idx, (x, y, _) in enumerate(loader):
#                 if b_idx >= MAX_BATCHES: break
#                 x, y = x.to(DEVICE), y.to(DEVICE)
#                 z_norm, logits = model[1](model[0](x))
#                 loss = F.cross_entropy(logits / 0.1, y) + margin_contrastive_loss(z_norm, y)
#                 opt.zero_grad(); loss.backward(); opt.step()
#                 if b_idx % 20 == 0: 
#                     print(f"       -> Stage 0 | Epoch {epoch} | Batch {b_idx} | Loss: {loss.item():.4f} | LR: {sched.get_last_lr()[0]:.4f}")
#             sched.step()
            
#         print("💾 Building Herding Memory & OOD Radii (Optimized One-Pass)...")
#         class_buckets = {c: [] for c in range(NUM_BASE_CLASSES)}
#         for b_idx, (x, y, _) in enumerate(loader):
#             if b_idx >= MAX_BATCHES * 5: break 
#             for i in range(len(y)):
#                 cls = y[i].item()
#                 if cls in class_buckets: class_buckets[cls].append(x[i])
                    
#         for cls, x_list in class_buckets.items():
#             if len(x_list) > 0:
#                 Xc = torch.stack(x_list)
#                 memory.build_memory_herding(Xc, cls, model, DEVICE)
#         detector.update(memory, model, DEVICE)

#     else:
#         print(f"🔍 [STAGE {stage}] Geometric OOD Discovery & HDBSCAN...")
#         model.eval()
#         novelty_buffer, candidate_stash = [], []
#         known_classes = model[1].last_layer.weight_g.shape[0]
        
#         with torch.no_grad():
#             for x, y, _ in loader:
#                 x = x.to(DEVICE)
#                 z_norm, _ = model[1](model[0](x))
#                 scores = [detector.score(z_norm[i]).item() for i in range(len(z_norm))]
#                 # thr = np.percentile(scores, 30)
#                 thr = np.percentile(scores, 75)
#                 for i in range(len(z_norm)):
#                     if scores[i] > thr: novelty_buffer.append((scores[i], x[i].cpu(), y[i].item()))
        
#         novelty_buffer.sort(reverse=True, key=lambda i: i[0])
#         novelty_buffer = [n for n in novelty_buffer[:400]]
        
#         if len(novelty_buffer) >= MNN_K + 1:
#             Z = []
#             with torch.no_grad():
#                 for _, img, _ in novelty_buffer:
#                     z_norm, _ = model[1](model[0](img.unsqueeze(0).to(DEVICE)))
#                     Z.append(z_norm.squeeze().cpu().numpy())
#             Z = np.stack(Z)
            
#             orig_dist = sp_dist.cdist(Z, Z, metric='cosine')
#             nn_model = NearestNeighbors(n_neighbors=MNN_K, metric='cosine').fit(Z)
#             _, indices = nn_model.kneighbors(Z)
            
#             mnn_dist = np.full((len(Z), len(Z)), 2.0); np.fill_diagonal(mnn_dist, 0.0)
#             for i in range(len(Z)):
#                 for j in indices[i]:
#                     if i in indices[j]: mnn_dist[i, j] = mnn_dist[j, i] = orig_dist[i, j]
                    
#             labels = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=MIN_CLUSTER_SIZE, cluster_selection_epsilon=0.0).fit_predict(mnn_dist)
            
#             for cid in sorted(set(labels)):
#                 idxs = np.where(labels == cid)[0]
#                 if cid == -1:
#                     noise_log.append(len(idxs))
#                     continue
                
#                 X_list = [novelty_buffer[i][1] for i in idxs]
#                 labels_true = [novelty_buffer[i][2] for i in idxs]
#                 Xc = torch.stack(X_list).to(DEVICE)
                
#                 # Calculate Ground Truth Purity
#                 most_common_cnt = Counter(labels_true).most_common(1)[0][1]
#                 cluster_purity = most_common_cnt / len(labels_true)
                
#                 with torch.no_grad(): z_norm, _ = model[1](model[0](Xc))
#                 mu = F.normalize(z_norm.mean(0), dim=0)
#                 S_intra = torch.mean(1 - torch.matmul(z_norm, mu)).item()
#                 S_known = min([1 - torch.dot(mu, detector.mu[k].to(DEVICE)).item() for k in detector.mu])
#                 density = len(idxs) / (S_intra + 1e-6)
#                 margin = S_known - S_intra
                
#                 if S_intra <= ALPHA and density >= DELTA and S_known >= BETA and margin > -0.30:
#                     candidate_stash.extend(X_list)
#                     # if len(candidate_stash) >= PROMOTION_THRESHOLD:
#                     if len(candidate_stash) >= PROMOTION_THRESHOLD and known_classes < 100:
#                         new_label = known_classes
#                         known_classes += 1
                        
#                         old_w = model[1].last_layer.weight.data.clone()
#                         old_n = old_w.shape[0]
#                         model[1] = DINOHead(in_dim=768, out_dim=known_classes).to(DEVICE)
#                         with torch.no_grad(): model[1].last_layer.weight.data[:old_n] = old_w
                        
#                         print(f"     🎉 [PROMOTION] Discovered New Class! Head expanded to {known_classes}.")
#                         model.train()
#                         # ft_opt = SGD([{'params': [p for p in model.parameters() if p.requires_grad]}], lr=0.01, momentum=0.9)
#                         ft_opt = SGD([{'params': [p for p in model.parameters() if p.requires_grad]}], lr=0.002, momentum=0.9)
                        
#                         # ---> DYNAMIC EPOCHS ONLINE FIX <---
#                         for ft_epoch in range(EPOCHS_ONLINE):
#                             X_mem, Y_mem = memory.sample_balanced(32)
#                             X_train = torch.stack(candidate_stash).to(DEVICE)
#                             Y_train = torch.full((len(X_train),), new_label, dtype=torch.long).to(DEVICE)
#                             if X_mem is not None:
#                                 X_train = torch.cat([X_train, X_mem.to(DEVICE)])
#                                 Y_train = torch.cat([Y_train, Y_mem.to(DEVICE)])
                                
#                             z_norm, logits = model[1](model[0](X_train))
#                             loss = F.cross_entropy(logits / 0.1, Y_train) + margin_contrastive_loss(z_norm, Y_train)
#                             ft_opt.zero_grad(); loss.backward(); ft_opt.step()
#                             print(f"       -> Finetune Epoch {ft_epoch} | Loss: {loss.item():.4f}")
                            
#                         memory.build_memory_herding(torch.stack(candidate_stash), new_label, model, DEVICE)
#                         detector.update(memory, model, DEVICE)
#                         discovery_log.append({'stage': stage, 'gt_labels': list(set(labels_true))})
#                         purity_log.append({'stage': stage, 'purity': cluster_purity, 'size': len(labels_true)})
#                         candidate_stash = []

#     # --- EVALUATION ---
#     print(f"📊 Evaluating Stage {stage} Accuracy...")
#     model.eval()
#     y_true, y_pred = [], []
#     eval_indices = ds.final_indices[:200] if RUN_MODE == "FAST_TEST" else ds.final_indices
    
#     with torch.no_grad():
#         for idx in eval_indices:
#             if ds.dataset_name == "C100": img = Image.fromarray(ds.image_paths_or_data[idx])
#             else: img = Image.open(ds.image_paths_or_data[idx]).convert('RGB')
#             img_t = standard_transform(img).unsqueeze(0).to(DEVICE)
#             _, logits = model[1](model[0](img_t))
#             y_true.append(ds.all_targets[idx])
#             y_pred.append(logits.argmax(1).item())
            
#     y_true, y_pred = np.array(y_true), np.array(y_pred)
#     mask = np.isin(y_true, ds.base_classes)
#     all_a, old_a, new_a = cluster_acc(y_true, y_pred, mask)
#     stage_metrics[stage] = {'all': all_a, 'old': old_a, 'new': new_a}
    
#     # Real-time Stage Output
#     print(f"   => Stage {stage} Results: All: {all_a:.2f}% | Old: {old_a:.2f}% | New: {new_a:.2f}%")

# # ==============================================================================
# # FINAL OUTPUTS
# # ==============================================================================
# print("\n" + "="*100)
# print(f"{'TABLE 1: STAGE-WISE CONTINUAL GCD ACCURACY (YOUR METHOD)':^100}")
# print("="*100)
# t1_str = f"{'Ours (Geom)':<15} | {stage_metrics[0]['all']:.2f} (All) | "
# for s in range(1, 6): t1_str += f"S{s}: {stage_metrics[s]['all']:.1f}/{stage_metrics[s]['old']:.1f}/{stage_metrics[s]['new']:.1f} | "
# print(t1_str)

# print("\n" + "="*50)
# print(f"{'TABLE 3: FORGETTING & DISCOVERY':^50}")
# print("="*50)
# M_f = stage_metrics[0]['old'] - stage_metrics[5]['old']
# M_d = stage_metrics[5]['new']
# print(f"{'Method':<15} | {'M_f ↓ (Forget)':<15} | {'M_d ↑ (Discover)':<15}")
# print(f"{'Ours (Geom)':<15} | {M_f:.2f}           | {M_d:.2f}")
# print("="*50)

# print("\n" + "🔥"*25)
# print("   YOUR DEEP DIVE DIAGNOSTICS")
# print("🔥"*25)
# print(f"1. Total Noise Filtered (OOD Rejections): {sum(noise_log)} sparse samples safely ignored.")
# print(f"2. Total Classes Discovered: {len(discovery_log)} classes promoted across 5 stages.")
# print("3. Discovery & Purity Timeline:")
# for d, p in zip(discovery_log, purity_log):
#     print(f"   -> Stage {d['stage']}: Purity: {p['purity']:.2f} (Size: {p['size']}) | Ground Truth Semantics Found: {d['gt_labels']}")

## dynamic updated cell 5

In [ ]:
# from torch.optim import SGD
# from sklearn.cluster import KMeans
# from sklearn.neighbors import NearestNeighbors
# from scipy.optimize import linear_sum_assignment
# import scipy.spatial.distance as sp_dist
# import hdbscan
# import copy
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from PIL import Image
# from collections import Counter

# # ==============================================================================
# # CELL 5: THE YOUR-METHOD PIPELINE (DYNAMIC + DEEP FREEZE + PURITY LOCK)
# # ==============================================================================
# DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# RUN_MODE = "1.5_HOUR_TEST"  
# TARGET_DATASET = "C100" 
# CURRENT_SPARSITY = 0.90

# # --- HUNGARIAN MATCHING ALGORITHM ---
# def cluster_acc(y_true, y_pred, mask):
#     y_true, y_pred, mask = y_true.astype(int), y_pred.astype(int), mask.astype(bool)
#     D = max(y_pred.max(), y_true.max()) + 1
#     w = np.zeros((D, D), dtype=int)
#     for i in range(y_pred.size): w[y_pred[i], y_true[i]] += 1
#     ind = np.vstack(linear_sum_assignment(w.max() - w)).T
#     ind_map = {j: i for i, j in ind}
#     total_acc = sum([w[i, j] for i, j in ind]) * 1.0 / y_pred.size
    
#     old_classes_gt, new_classes_gt = set(y_true[mask]), set(y_true[~mask])
#     old_acc, total_old = 0, 0
#     for i in old_classes_gt:
#         old_acc += w[ind_map[i], i] if i in ind_map else 0
#         total_old += sum(w[:, i])
#     old_acc = (old_acc / total_old) * 100 if total_old > 0 else 0.0

#     new_acc, total_new = 0, 0
#     for i in new_classes_gt:
#         new_acc += w[ind_map[i], i] if i in ind_map else 0
#         total_new += sum(w[:, i])
#     new_acc = (new_acc / total_new) * 100 if total_new > 0 else 0.0
#     return total_acc * 100, old_acc, new_acc

# # --- THE PURITY LOCK PARAMETERS (Partially Dynamic) ---
# ALPHA = 0.40             # Extremely strict intra-cluster distance. Must be tight!
# DELTA = 5                # Demand higher density
# MIN_CLUSTER_SIZE = 5     # Ignore tiny 3-image noise balls
# MNN_K = 5                
# MERGE_THRESHOLD = 0.90   

# # These will be dynamically overwritten in the code, but we set safe defaults here
# BETA = 0.05              
# PROMOTION_THRESHOLD = 20 

# DATASET_CONFIGS = {"C100": {"base": 50, "chunk": 10}, "IN100": {"base": 50, "chunk": 10}, "TINY": {"base": 100, "chunk": 20}, "CUB": {"base": 100, "chunk": 20}}
# NUM_BASE_CLASSES = DATASET_CONFIGS[TARGET_DATASET]["base"]

# # HARDCODED SPEEDRUN LIMITS (< 1.5 Hour Guarantee)
# if RUN_MODE == "2_HOUR_TEST": 
#     EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 10, 5, 999999
# elif RUN_MODE == "1.5_HOUR_TEST": 
#     EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 5, 2, 999999
# elif RUN_MODE == "FAST_TEST": 
#     EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 1, 1, 3
# else: 
#     EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 100, 30, 999999

# # Init DINO
# print("Initializing DINO ViT Backbone...")
# backbone = torch.hub.load('facebookresearch/dino:main', 'dino_vitb16', verbose=False)
# for m in backbone.parameters(): m.requires_grad = False
# for name, m in backbone.named_parameters():
#     if 'blocks.11' in name: m.requires_grad = True

# model = nn.Sequential(backbone, DINOHead(in_dim=768, out_dim=NUM_BASE_CLASSES)).to(DEVICE)
# memory = MemoryBuffer(max_per_class=100)
# detector = HypersphereNovelty()

# stage_metrics = {}
# discovery_log = [] 
# noise_log = []     
# purity_log = []

# print("\n" + "="*80)
# print(f"🚀 STARTING YOUR GEOMETRIC OOD PIPELINE: LOCKDOWN SPEEDRUN (DYNAMIC)")
# print(f"⚙️ Config Locked: Offline Epochs: {EPOCHS_OFFLINE} | Online Epochs: {EPOCHS_ONLINE} | Sparsity: {CURRENT_SPARSITY}")
# print("="*80)

# for stage in range(6):
#     print(f"\n---> INITIALIZING STAGE {stage} <---")
#     ds = CGCD_Master_Dataset(TARGET_DATASET, DATA_PATHS[TARGET_DATASET], stage=stage, sparsity_level=CURRENT_SPARSITY)
#     loader = torch.utils.data.DataLoader(GeometricDatasetWrapper(ds, standard_transform), batch_size=128, drop_last=False, shuffle=True)
    
#     # ---> DYNAMIC AUTO-CALIBRATOR: PROMOTION THRESHOLD <---
#     # CIFAR100 has 500 train images per class natively. Adjust this base number if using ImageNet/CUB.
#     expected_imgs_per_class = 500 * (1.0 - CURRENT_SPARSITY) 
#     PROMOTION_THRESHOLD = max(5, int(expected_imgs_per_class * 0.40)) # Require 40% of available class data
#     if stage > 0:
#         print(f"   ⚙️ [DYNAMIC CALIBRATION] Expected Imgs/Class: {expected_imgs_per_class:.0f} -> PROMOTION_THRESHOLD locked at {PROMOTION_THRESHOLD}")

#     if stage == 0:
#         print("🎓 [STAGE 0] Training Base Classes...")
#         model.train()
#         opt = SGD([{'params': [p for p in model.parameters() if p.requires_grad]}], lr=0.1, momentum=0.9, weight_decay=5e-5)
#         sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_OFFLINE)
        
#         for epoch in range(EPOCHS_OFFLINE):
#             for b_idx, (x, y, _) in enumerate(loader):
#                 if b_idx >= MAX_BATCHES: break
#                 x, y = x.to(DEVICE), y.to(DEVICE)
#                 z_norm, logits = model[1](model[0](x))
#                 loss = F.cross_entropy(logits / 0.1, y) + margin_contrastive_loss(z_norm, y)
#                 opt.zero_grad(); loss.backward(); opt.step()
#                 if b_idx % 20 == 0: 
#                     print(f"       -> Stage 0 | Epoch {epoch} | Batch {b_idx} | Loss: {loss.item():.4f} | LR: {sched.get_last_lr()[0]:.4f}")
#             sched.step()
            
#         print("💾 Building Herding Memory & OOD Radii (Optimized One-Pass)...")
#         class_buckets = {c: [] for c in range(NUM_BASE_CLASSES)}
#         for b_idx, (x, y, _) in enumerate(loader):
#             if b_idx >= MAX_BATCHES * 5: break 
#             for i in range(len(y)):
#                 cls = y[i].item()
#                 if cls in class_buckets: class_buckets[cls].append(x[i])
                    
#         for cls, x_list in class_buckets.items():
#             if len(x_list) > 0:
#                 Xc = torch.stack(x_list)
#                 memory.build_memory_herding(Xc, cls, model, DEVICE)
#         detector.update(memory, model, DEVICE)
        
#         # ---> DYNAMIC AUTO-CALIBRATOR: BETA (GEOMETRIC BOUNCER) <---
#         if len(detector.mu) > 1:
#             mu_tensor = torch.stack([detector.mu[k] for k in detector.mu]).to(DEVICE)
#             sim_matrix = torch.matmul(mu_tensor, mu_tensor.T)
#             avg_base_distance = (1 - sim_matrix).mean().item()
#             BETA = avg_base_distance * 0.10 # Require new classes to be at least 10% of avg distance away
#             print(f"   ⚙️ [DYNAMIC CALIBRATION] Base Distance: {avg_base_distance:.4f} -> BETA locked at {BETA:.4f}")

#     else:
#         print(f"🔍 [STAGE {stage}] Geometric OOD Discovery & HDBSCAN...")
#         model.eval()
#         novelty_buffer, candidate_stash = [], []
#         known_classes = model[1].last_layer.weight_g.shape[0]
        
#         with torch.no_grad():
#             for x, y, _ in loader:
#                 x = x.to(DEVICE)
#                 z_norm, _ = model[1](model[0](x))
#                 scores = [detector.score(z_norm[i]).item() for i in range(len(z_norm))]
                
#                 # CRITICAL FIX: Only let the top 15% weirdest images in (starve the noise)
#                 thr = np.percentile(scores, 85) 
                
#                 for i in range(len(z_norm)):
#                     if scores[i] > thr: novelty_buffer.append((scores[i], x[i].cpu(), y[i].item()))
        
#         novelty_buffer.sort(reverse=True, key=lambda i: i[0])
#         novelty_buffer = [n for n in novelty_buffer[:400]]
        
#         if len(novelty_buffer) >= MNN_K + 1:
#             Z = []
#             with torch.no_grad():
#                 for _, img, _ in novelty_buffer:
#                     z_norm, _ = model[1](model[0](img.unsqueeze(0).to(DEVICE)))
#                     Z.append(z_norm.squeeze().cpu().numpy())
#             Z = np.stack(Z)
            
#             orig_dist = sp_dist.cdist(Z, Z, metric='cosine')
#             nn_model = NearestNeighbors(n_neighbors=MNN_K, metric='cosine').fit(Z)
#             _, indices = nn_model.kneighbors(Z)
            
#             mnn_dist = np.full((len(Z), len(Z)), 2.0); np.fill_diagonal(mnn_dist, 0.0)
#             for i in range(len(Z)):
#                 for j in indices[i]:
#                     if i in indices[j]: mnn_dist[i, j] = mnn_dist[j, i] = orig_dist[i, j]
                    
#             labels = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=MIN_CLUSTER_SIZE, cluster_selection_epsilon=0.0).fit_predict(mnn_dist)
            
#             for cid in sorted(set(labels)):
#                 idxs = np.where(labels == cid)[0]
#                 if cid == -1:
#                     noise_log.append(len(idxs))
#                     continue
                
#                 X_list = [novelty_buffer[i][1] for i in idxs]
#                 labels_true = [novelty_buffer[i][2] for i in idxs]
#                 Xc = torch.stack(X_list).to(DEVICE)
                
#                 most_common_cnt = Counter(labels_true).most_common(1)[0][1]
#                 cluster_purity = most_common_cnt / len(labels_true)
                
#                 with torch.no_grad(): z_norm, _ = model[1](model[0](Xc))
#                 mu = F.normalize(z_norm.mean(0), dim=0)
#                 S_intra = torch.mean(1 - torch.matmul(z_norm, mu)).item()
#                 S_known = min([1 - torch.dot(mu, detector.mu[k].to(DEVICE)).item() for k in detector.mu])
#                 density = len(idxs) / (S_intra + 1e-6)
#                 margin = S_known - S_intra
                
#                 if S_intra <= ALPHA and density >= DELTA and S_known >= BETA and margin > -0.30:
#                     candidate_stash.extend(X_list)
                    
#                     # CRITICAL FIX: Hard Cap to prevent runaway memory explosion
#                     if len(candidate_stash) >= PROMOTION_THRESHOLD and known_classes < 100:
#                         new_label = known_classes
#                         known_classes += 1
                        
#                         old_w = model[1].last_layer.weight.data.clone()
#                         old_n = old_w.shape[0]
#                         model[1] = DINOHead(in_dim=768, out_dim=known_classes).to(DEVICE)
#                         with torch.no_grad(): model[1].last_layer.weight.data[:old_n] = old_w
                        
#                         print(f"     🎉 [PROMOTION] Discovered New Class! Head expanded to {known_classes}.")
#                         model.train()
                        
#                         # ---> THE DEEP FREEZE FIX <---
#                         # Lock the backbone completely so memory CANNOT be destroyed
#                         for param in model[0].parameters(): param.requires_grad = False
                        
#                         # Only train the projection head on the new class with a lower learning rate
#                         ft_opt = SGD(model[1].parameters(), lr=0.005, momentum=0.9)
                        
#                         for ft_epoch in range(EPOCHS_ONLINE):
#                             X_mem, Y_mem = memory.sample_balanced(32)
#                             X_train = torch.stack(candidate_stash).to(DEVICE)
#                             Y_train = torch.full((len(X_train),), new_label, dtype=torch.long).to(DEVICE)
#                             if X_mem is not None:
#                                 X_train = torch.cat([X_train, X_mem.to(DEVICE)])
#                                 Y_train = torch.cat([Y_train, Y_mem.to(DEVICE)])
                                
#                             z_norm, logits = model[1](model[0](X_train))
#                             loss = F.cross_entropy(logits / 0.1, Y_train) + margin_contrastive_loss(z_norm, Y_train)
#                             ft_opt.zero_grad(); loss.backward(); ft_opt.step()
#                             print(f"       -> Finetune Epoch {ft_epoch} | Loss: {loss.item():.4f}")
                            
#                         memory.build_memory_herding(torch.stack(candidate_stash), new_label, model, DEVICE)
#                         detector.update(memory, model, DEVICE)
#                         discovery_log.append({'stage': stage, 'gt_labels': list(set(labels_true))})
#                         purity_log.append({'stage': stage, 'purity': cluster_purity, 'size': len(labels_true)})
#                         candidate_stash = []

#     # --- EVALUATION ---
#     print(f"📊 Evaluating Stage {stage} Accuracy...")
#     model.eval()
#     y_true, y_pred = [], []
#     eval_indices = ds.final_indices[:200] if RUN_MODE == "FAST_TEST" else ds.final_indices
    
#     with torch.no_grad():
#         for idx in eval_indices:
#             if ds.dataset_name == "C100": img = Image.fromarray(ds.image_paths_or_data[idx])
#             else: img = Image.open(ds.image_paths_or_data[idx]).convert('RGB')
#             img_t = standard_transform(img).unsqueeze(0).to(DEVICE)
#             _, logits = model[1](model[0](img_t))
#             y_true.append(ds.all_targets[idx])
#             y_pred.append(logits.argmax(1).item())
            
#     y_true, y_pred = np.array(y_true), np.array(y_pred)
#     mask = np.isin(y_true, ds.base_classes)
#     all_a, old_a, new_a = cluster_acc(y_true, y_pred, mask)
#     stage_metrics[stage] = {'all': all_a, 'old': old_a, 'new': new_a}
    
#     print(f"   => Stage {stage} Results: All: {all_a:.2f}% | Old: {old_a:.2f}% | New: {new_a:.2f}%")

# # ==============================================================================
# # FINAL OUTPUTS
# # ==============================================================================
# print("\n" + "="*100)
# print(f"{'TABLE 1: STAGE-WISE CONTINUAL GCD ACCURACY (YOUR METHOD)':^100}")
# print("="*100)
# t1_str = f"{'Ours (Geom)':<15} | {stage_metrics[0]['all']:.2f} (All) | "
# for s in range(1, 6): t1_str += f"S{s}: {stage_metrics[s]['all']:.1f}/{stage_metrics[s]['old']:.1f}/{stage_metrics[s]['new']:.1f} | "
# print(t1_str)

# print("\n" + "="*50)
# print(f"{'TABLE 3: FORGETTING & DISCOVERY':^50}")
# print("="*50)
# M_f = stage_metrics[0]['old'] - stage_metrics[5]['old']
# M_d = stage_metrics[5]['new']
# print(f"{'Method':<15} | {'M_f ↓ (Forget)':<15} | {'M_d ↑ (Discover)':<15}")
# print(f"{'Ours (Geom)':<15} | {M_f:.2f}           | {M_d:.2f}")
# print("="*50)

# print("\n" + "🔥"*25)
# print("   YOUR DEEP DIVE DIAGNOSTICS")
# print("🔥"*25)
# print(f"1. Total Noise Filtered (OOD Rejections): {sum(noise_log)} sparse samples safely ignored.")
# print(f"2. Total Classes Discovered: {len(discovery_log)} classes promoted across 5 stages.")
# print("3. Discovery & Purity Timeline:")
# for d, p in zip(discovery_log, purity_log):
#     print(f"   -> Stage {d['stage']}: Purity: {p['purity']:.2f} (Size: {p['size']}) | Ground Truth Semantics Found: {d['gt_labels']}")

## CELL 5: The "Evidence Accumulation" Pipeline 

In [ ]:
from torch.optim import SGD
from sklearn.cluster import AgglomerativeClustering
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from collections import Counter
import scipy.spatial.distance as sp_dist

# ==============================================================================
# CELL 5: THE YOUR-METHOD PIPELINE (EVIDENCE-ACCUMULATION + DEEP DIAGNOSTICS)
# ==============================================================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RUN_MODE = "1.5_HOUR_TEST"  
TARGET_DATASET = "C100" 
CURRENT_SPARSITY = 0.90

# --- HUNGARIAN MATCHING ALGORITHM ---
def cluster_acc(y_true, y_pred, mask):
    from scipy.optimize import linear_sum_assignment
    y_true, y_pred, mask = y_true.astype(int), y_pred.astype(int), mask.astype(bool)
    D = max(y_pred.max(), y_true.max()) + 1
    w = np.zeros((D, D), dtype=int)
    for i in range(y_pred.size): w[y_pred[i], y_true[i]] += 1
    ind = np.vstack(linear_sum_assignment(w.max() - w)).T
    ind_map = {j: i for i, j in ind}
    
    total_acc = sum([w[i, j] for i, j in ind]) * 1.0 / y_pred.size
    
    old_classes_gt, new_classes_gt = set(y_true[mask]), set(y_true[~mask])
    old_acc, total_old = 0, 0
    for i in old_classes_gt:
        old_acc += w[ind_map[i], i] if i in ind_map else 0
        total_old += sum(w[:, i])
    old_acc = (old_acc / total_old) * 100 if total_old > 0 else 0.0

    new_acc, total_new = 0, 0
    for i in new_classes_gt:
        new_acc += w[ind_map[i], i] if i in ind_map else 0
        total_new += sum(w[:, i])
    new_acc = (new_acc / total_new) * 100 if total_new > 0 else 0.0
    return total_acc * 100, old_acc, new_acc

# --- THE NEW EVIDENCE PARAMETERS ---
ALPHA_DIST = 0.15          # Max intra-cluster cosine distance (Strict Proximity)
PROMOTION_MIN_SIZE = 8     # Minimum cluster size required for promotion (Tolerates 90% Sparsity)

DATASET_CONFIGS = {"C100": {"base": 50}, "IN100": {"base": 50}, "TINY": {"base": 100}, "CUB": {"base": 100}}
NUM_BASE_CLASSES = DATASET_CONFIGS[TARGET_DATASET]["base"]

# HARDCODED SPEEDRUN LIMITS (< 1.5 Hour Guarantee)
if RUN_MODE == "2_HOUR_TEST": 
    EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 10, 5, 999999
elif RUN_MODE == "1.5_HOUR_TEST": 
    EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 5, 2, 999999
elif RUN_MODE == "FAST_TEST": 
    EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 1, 1, 3
else: 
    EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 100, 30, 999999

# Init DINO
print("Initializing DINO ViT Backbone...")
backbone = torch.hub.load('facebookresearch/dino:main', 'dino_vitb16', verbose=False)
for m in backbone.parameters(): m.requires_grad = False
for name, m in backbone.named_parameters():
    if 'blocks.11' in name: m.requires_grad = True

model = nn.Sequential(backbone, DINOHead(in_dim=768, out_dim=NUM_BASE_CLASSES)).to(DEVICE)
memory = MemoryBuffer(max_per_class=100)
detector = HypersphereNovelty(q=0.95)

# DIAGNOSTIC LOGS
stage_metrics = {}
discovery_log = [] 
noise_log = []     
purity_log = []

print("\n" + "="*80)
print(f"🚀 STARTING YOUR GEOMETRIC OOD PIPELINE: EVIDENCE ACCUMULATION (SPARSITY {CURRENT_SPARSITY})")
print(f"⚙️ Config Locked: Offline Epochs: {EPOCHS_OFFLINE} | Online Epochs: {EPOCHS_ONLINE}")
print("="*80)

for stage in range(6):
    print(f"\n---> INITIALIZING STAGE {stage} <---")
    ds = CGCD_Master_Dataset(TARGET_DATASET, DATA_PATHS[TARGET_DATASET], stage=stage, sparsity_level=CURRENT_SPARSITY)
    loader = torch.utils.data.DataLoader(GeometricDatasetWrapper(ds, standard_transform), batch_size=128, drop_last=False, shuffle=True)

    if stage == 0:
        print("🎓 [STAGE 0] Training Base Classes...")
        model.train()
        opt = SGD([{'params': [p for p in model.parameters() if p.requires_grad]}], lr=0.1, momentum=0.9, weight_decay=5e-5)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_OFFLINE)
        
        for epoch in range(EPOCHS_OFFLINE):
            for b_idx, (x, y, _) in enumerate(loader):
                if b_idx >= MAX_BATCHES: break
                x, y = x.to(DEVICE), y.to(DEVICE)
                z_norm, logits = model[1](model[0](x))
                loss = F.cross_entropy(logits / 0.1, y) + margin_contrastive_loss(z_norm, y)
                opt.zero_grad(); loss.backward(); opt.step()
                if b_idx % 20 == 0: 
                    print(f"       -> Stage 0 | Epoch {epoch} | Batch {b_idx} | Loss: {loss.item():.4f} | LR: {sched.get_last_lr()[0]:.4f}")
            sched.step()
            
        print("💾 Building Herding Memory & Base Geometry...")
        class_buckets = {c: [] for c in range(NUM_BASE_CLASSES)}
        for b_idx, (x, y, _) in enumerate(loader):
            if b_idx >= MAX_BATCHES * 5: break 
            for i in range(len(y)):
                if len(class_buckets[y[i].item()]) < 100: class_buckets[y[i].item()].append(x[i])
                    
        for cls, x_list in class_buckets.items():
            if len(x_list) > 0: memory.build_memory_herding(torch.stack(x_list), cls, model, DEVICE)
        detector.update(memory, model, DEVICE)

    else:
        print(f"🔍 [STAGE {stage}] Geometric Evidence Accumulation...")
        model.eval()
        z_novel, x_novel, y_novel = [], [], []
        known_classes = model[1].last_layer.out_features
        
        # 1. ABSOLUTE OOD GATING
        with torch.no_grad():
            for x, y, _ in loader:
                x_dev = x.to(DEVICE)
                z_norm, _ = model[1](model[0](x_dev))
                for i in range(len(z_norm)):
                    score = detector.score(z_norm[i]).item()
                    if score > -0.05: 
                        z_novel.append(z_norm[i].cpu().numpy())
                        x_novel.append(x[i])
                        y_novel.append(y[i].item())
        
        if len(z_novel) >= PROMOTION_MIN_SIZE:
            Z = np.array(z_novel)
            
            # 2. PROXIMITY-BASED CLUSTERING
            clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=ALPHA_DIST, metric='cosine', linkage='average')
            cluster_labels = clustering.fit_predict(Z)
            
            counts = Counter(cluster_labels)
            for cid, count in counts.items():
                idxs = np.where(cluster_labels == cid)[0]
                
                if count < PROMOTION_MIN_SIZE:
                    noise_log.append(count)
                    continue
                
                labels_true = [y_novel[i] for i in idxs]
                most_common_cnt = Counter(labels_true).most_common(1)[0][1]
                cluster_purity = most_common_cnt / len(labels_true)
                
                cluster_mu = torch.from_numpy(Z[idxs].mean(0)).to(DEVICE)
                cluster_mu = F.normalize(cluster_mu, dim=0)
                
                # 3. PROTOTYPICAL HEAD INJECTION (FIXED DIMENSIONS & MLP RETENTION)
                new_label = known_classes
                known_classes += 1
                
                old_head = model[1]
                
                # The backbone output is 768. The bottleneck output is 256.
                new_head = DINOHead(in_dim=768, out_dim=known_classes).to(DEVICE)
                
                with torch.no_grad():
                    # CRITICAL FIX: Retain the MLP projection geometry
                    new_head.mlp.load_state_dict(old_head.mlp.state_dict())
                    
                    # DINOHead uses weight_norm, so we must map weight_v and weight_g explicitly
                    old_v = old_head.last_layer.weight_v.data
                    old_g = old_head.last_layer.weight_g.data
                    
                    new_head.last_layer.weight_v.data[:known_classes-1] = old_v
                    new_head.last_layer.weight_g.data[:known_classes-1] = old_g
                    
                    # Inject the new cluster prototype
                    new_head.last_layer.weight_v.data[known_classes-1] = cluster_mu
                    new_head.last_layer.weight_g.data[known_classes-1] = 1.0
                
                model[1] = new_head
                print(f"     🎉 [PROMOTION] Discovered New Class! Head expanded to {known_classes}.")
                
                # 4. TARGETED ONLINE ALIGNMENT
                model.train()
                for param in model[0].parameters(): param.requires_grad = False
                ft_opt = SGD(model[1].parameters(), lr=0.005, momentum=0.9)
                
                X_train = torch.stack([x_novel[i] for i in idxs]).to(DEVICE)
                Y_train = torch.full((len(idxs),), new_label, dtype=torch.long).to(DEVICE)
                
                for ft_epoch in range(EPOCHS_ONLINE):
                    X_mem, Y_mem = memory.sample_balanced(32)
                    if X_mem is not None:
                        bx, by = torch.cat([X_train, X_mem.to(DEVICE)]), torch.cat([Y_train, Y_mem.to(DEVICE)])
                    else: bx, by = X_train, Y_train
                        
                    z_norm, logits = model[1](model[0](bx))
                    loss = F.cross_entropy(logits / 0.1, by) + margin_contrastive_loss(z_norm, by)
                    ft_opt.zero_grad(); loss.backward(); ft_opt.step()
                    print(f"       -> Finetune Epoch {ft_epoch} | Loss: {loss.item():.4f}")
                
                # Log Diagnostics & Update Memory
                memory.build_memory_herding(X_train, new_label, model, DEVICE)
                detector.update(memory, model, DEVICE)
                discovery_log.append({'stage': stage, 'gt_labels': list(set(labels_true))})
                purity_log.append({'stage': stage, 'purity': cluster_purity, 'size': len(labels_true)})
                
        else:
            noise_log.append(len(z_novel))

    # --- EVALUATION ---
    print(f"📊 Evaluating Stage {stage} Accuracy...")
    model.eval()
    y_true, y_pred = [], []
    eval_indices = ds.final_indices[:200] if RUN_MODE == "FAST_TEST" else ds.final_indices
    
    with torch.no_grad():
        for idx in eval_indices:
            if ds.dataset_name == "C100": img = Image.fromarray(ds.image_paths_or_data[idx])
            else: img = Image.open(ds.image_paths_or_data[idx]).convert('RGB')
            img_t = standard_transform(img).unsqueeze(0).to(DEVICE)
            _, logits = model[1](model[0](img_t))
            y_true.append(ds.all_targets[idx])
            y_pred.append(logits.argmax(1).item())
            
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = np.isin(y_true, ds.base_classes)
    all_a, old_a, new_a = cluster_acc(y_true, y_pred, mask)
    stage_metrics[stage] = {'all': all_a, 'old': old_a, 'new': new_a}
    
    print(f"   => Stage {stage} Results: All: {all_a:.2f}% | Old: {old_a:.2f}% | New: {new_a:.2f}%")

# ==============================================================================
# FINAL OUTPUTS (EXACT ORIGINAL FORMATTING)
# ==============================================================================
print("\n" + "="*100)
print(f"{'TABLE 1: STAGE-WISE CONTINUAL GCD ACCURACY (YOUR METHOD)':^100}")
print("="*100)
t1_str = f"{'Ours (Geom)':<15} | {stage_metrics[0]['all']:.2f} (All) | "
for s in range(1, 6): t1_str += f"S{s}: {stage_metrics[s]['all']:.1f}/{stage_metrics[s]['old']:.1f}/{stage_metrics[s]['new']:.1f} | "
print(t1_str)

print("\n" + "="*50)
print(f"{'TABLE 3: FORGETTING & DISCOVERY':^50}")
print("="*50)
M_f = stage_metrics[0]['old'] - stage_metrics[5]['old']
M_d = stage_metrics[5]['new']
print(f"{'Method':<15} | {'M_f ↓ (Forget)':<15} | {'M_d ↑ (Discover)':<15}")
print(f"{'Ours (Geom)':<15} | {M_f:.2f}           | {M_d:.2f}")
print("="*50)

print("\n" + "🔥"*25)
print("   YOUR DEEP DIVE DIAGNOSTICS")
print("🔥"*25)
print(f"1. Total Noise Filtered (OOD Rejections): {sum(noise_log)} sparse samples safely ignored.")
print(f"2. Total Classes Discovered: {len(discovery_log)} classes promoted across 5 stages.")
print("3. Discovery & Purity Timeline:")
for d, p in zip(discovery_log, purity_log):
    print(f"   -> Stage {d['stage']}: Purity: {p['purity']:.2f} (Size: {p['size']}) | Ground Truth Semantics Found: {d['gt_labels']}")